# A6: Speech Processing

In this lab, we cover the full speech pipeline with two central themes:

1. **How speech is tokenized** — and why it is fundamentally different from NLP text tokenization
2. **Emotional TTS** — synthesizing speech with controllable emotions using emotion tags

---
## The Two Core Tasks

| Task | Direction | Key Challenge |
|---|---|---|
| **ASR** | Audio → Text | Align continuous audio frames to discrete tokens |
| **TTS** | Text → Audio | Map discrete tokens back to a continuous, expressive waveform |

Both tasks go through the same bridge: the **Mel Spectrogram** — a 2D time-frequency representation that compresses raw audio into a form neural networks can process.

---

# ⚙️ Setup & Dependencies — A6

**Run this cell first** before starting the lab. It will automatically install any missing packages.

| Package | Min Version | Purpose |
|---|---|---|
| `torch` | `>=2.0` | Deep learning framework |
| `torchaudio` | `>=2.0` | Audio loading + Mel Spectrogram |
| `numpy` | `>=1.24` | Numerical computing |
| `matplotlib` | `>=3.7` | Spectrogram visualization |
| `pandas` | `>=2.0` | RAVDESS metadata |
| `Pillow` | `>=9.0` | Image processing |
| `scipy` | `>=1.11` | Audio file writing (BARK output) |
| `TTS` | `>=0.22` | Coqui-TTS for synthesis |

**Optional packages** (only needed for specific exercises):

| Package | Required | Purpose |
|---|---|---|
| `g2p_en` | optional | Grapheme-to-phoneme conversion (Exercise 1) |
| `bark` | optional | BARK generative TTS (Section 3.5, needs ~5GB GPU RAM) |

> 💡 **Google Colab users:** Most packages are pre-installed. Only `timm`, `TTS`, `g2p_en`, `bark` may need installing.

> 💻 **Local users:** If you use conda, replace `pip install` with `conda install` where available.

---

In [ ]:
# ── Install missing packages ─────────────────────────────────────────────
import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

required = ['torch', 'torchaudio', 'numpy', 'matplotlib', 'pandas', 'Pillow', 'scipy', 'TTS']
for pkg in required:
    try:
        __import__(pkg.replace('-','_').split('==')[0])
    except ImportError:
        print(f'Installing {pkg}...')
        install(pkg)


# ── Optional (uncomment if needed) ──────────────────────────────────────
# pip install g2p_en  # Grapheme-to-phoneme conversion (Exercise 1)
# pip install bark  # BARK generative TTS (Section 3.5, needs ~5GB GPU RAM)

# ── Verify all imports work ───────────────────────────────────────────────
print(f'\nA6 dependencies check:')
try:
    import torch; print(f'  ✅ torch {getattr(torch, "__version__", "ok")}')
    import torchaudio; print(f'  ✅ torchaudio {getattr(torchaudio, "__version__", "ok")}')
    import numpy; print(f'  ✅ numpy {getattr(numpy, "__version__", "ok")}')
    import matplotlib; print(f'  ✅ matplotlib {getattr(matplotlib, "__version__", "ok")}')
    import pandas; print(f'  ✅ pandas {getattr(pandas, "__version__", "ok")}')
    print('\n✅ All dependencies satisfied — ready to start!')
except ImportError as e:
    print(f'\n❌ Missing: {e}')
    print('Run: pip install ' + str(e).split("'")[1])

## 📚 Papers & Code References

| Model | Paper | Venue | Code Base |
|---|---|---|---|
| **Mel Spectrogram** | Davis & Mermelstein (1980). *Comparison of Parametric Representations for Word Recognition* | IEEE TASLP | `torchaudio.transforms.MelSpectrogram` |
| **WaveNet** | van den Oord et al. (2016). *WaveNet: A Generative Model for Raw Audio* | arXiv 2016 | Background / conceptual |
| **Tacotron** | Wang et al. (2017). *Tacotron: Towards End-to-End Speech Synthesis* | Interspeech 2017 | Coqui-TTS: `tts_models/en/ljspeech/tacotron2-DDC` |
| **Tacotron 2** | Shen et al. (2018). *Natural TTS Synthesis by Conditioning WaveNet on Mel Spectrograms* | ICASSP 2018 | Same Coqui-TTS model |
| **WaveRNN** | Kalchbrenner et al. (2018). *Efficient Neural Audio Synthesis* | ICML 2018 | Background only |
| **Emotion TTS** | Li et al. (2022). *StyleTTS: Style-Based Generative Model for TTS* | NeurIPS 2022 | Conceptual basis for emotion tags |
| **RAVDESS** | Livingstone & Russo (2018). *The Ryerson Audio-Visual Database of Emotional Speech and Song* | PLOS ONE 2018 | Downloaded from Zenodo (open access, CC BY-NA-SC 4.0) |
| **BARK** | Suno (2023). *Bark: Text-Prompted Generative Audio Model* | — | `pip install bark` |

**External code used:**
- TTS synthesis: Coqui-TTS https://github.com/coqui-ai/TTS (MPL-2.0)
- `SpeechTokenizer` class: written from scratch for this lab
- Emotion post-processing: custom implementation inspired by BARK prompt engineering

**Paper links:**
- WaveNet: https://arxiv.org/abs/1609.03499
- Tacotron: https://arxiv.org/abs/1703.10135
- Tacotron 2: https://arxiv.org/abs/1712.05884
- StyleTTS: https://arxiv.org/abs/2205.15439
- RAVDESS: https://zenodo.org/record/1188976
- BARK: https://github.com/suno-ai/bark

---

In [ ]:
import torch
import torchaudio
import torchaudio.transforms as T
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Audio, display
import os, re

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
os.makedirs('data/speech', exist_ok=True)

---
# Part 1: Speech Tokenization — Why It's Different from NLP

## 1.1 How NLP Tokenizes Text

In NLP (e.g., BERT, GPT), tokenization is conceptually simple:

```
"Hello world" → ["Hello", "world"] → [7592, 2088]   # word-level
"Hello world" → ["He", "llo", "▁world"] → [1544, 7414, 2088]  # BPE (subword)
```

- **Discrete**: text IS already a sequence of discrete symbols
- **Fixed vocabulary**: 30,000–50,000 tokens, every token is equally "sized"
- **No alignment problem**: token boundaries are explicit (spaces, subword splits)
- **One token = one meaning unit** (roughly)

## 1.2 Why Audio Is Fundamentally Different

Audio is a **continuous** signal sampled at 16,000–44,100 points per second. There are **no natural boundaries** between sounds.

```
"Hello" spoken aloud:
→ 32,000 raw sample values (2 seconds at 16kHz)
→ No spaces, no punctuation, no clear cut points
→ The same word spoken fast vs slow has VERY different lengths
```

**The core problem:** How do you tokenize something continuous?

In [ ]:
# ── Visualize: NLP token vs Audio signal side by side ──────────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 4))

# Left: NLP tokenization — discrete, uniform
ax = axes[0]
tokens = ['He', '##llo', 'world', '!']
token_ids = [7592, 7414, 2088, 999]
colors = ['#4C72B0','#DD8452','#55A868','#C44E52']
for i, (tok, tid, col) in enumerate(zip(tokens, token_ids, colors)):
    ax.barh(0, 1, left=i, height=0.5, color=col, edgecolor='white', linewidth=2)
    ax.text(i + 0.5, 0, f'{tok}\n[{tid}]', ha='center', va='center',
            fontsize=11, fontweight='bold', color='white')
ax.set_xlim(0, 4); ax.set_ylim(-0.5, 0.5)
ax.set_yticks([]); ax.set_xticks([])
ax.set_title('NLP Tokenization\nDiscrete, uniform, explicit boundaries', fontsize=12)
ax.set_xlabel('Each bar = 1 token, always the same "size"')

# Right: audio signal — continuous, no boundaries
ax2 = axes[1]
t = np.linspace(0, 0.5, 8000)
# Simulate a rough "Hello" envelope
signal = (np.sin(2*np.pi*300*t) * np.exp(-10*(t-0.05)**2) * 0.8 +
          np.sin(2*np.pi*500*t) * np.exp(-15*(t-0.2)**2) * 0.6 +
          np.sin(2*np.pi*200*t) * np.exp(-20*(t-0.35)**2) * 0.9)
signal += np.random.randn(len(t)) * 0.05
ax2.plot(t * 1000, signal, color='steelblue', linewidth=0.5)
ax2.axvspan(10, 80,   alpha=0.15, color='red',    label='"H"')
ax2.axvspan(80, 200,  alpha=0.15, color='green',  label='"ell"')
ax2.axvspan(200,370,  alpha=0.15, color='orange', label='"o"')
ax2.set_title('Audio Signal ("Hello" spoken)\nContinuous, no explicit boundaries', fontsize=12)
ax2.set_xlabel('Time (ms)'); ax2.set_ylabel('Amplitude')
ax2.legend(loc='upper right', fontsize=9)
ax2.text(45,  0.9, '"H"?',  ha='center', fontsize=10, color='red')
ax2.text(140, 0.9, '"ell"?', ha='center', fontsize=10, color='green')
ax2.text(285, 0.9, '"o"?',  ha='center', fontsize=10, color='orange')

plt.suptitle('NLP vs Speech: The Tokenization Problem', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

print('Key difference:')
print('NLP  → text already IS tokens → just look up in vocabulary')
print('Audio → must DECIDE where tokens are → no obvious boundaries')

## 1.3 The Three Approaches to Speech Tokenization

Because audio has no natural token boundaries, researchers invented three different strategies:

### Approach 1: Character/Grapheme Tokens (Tacotron style)
Tokenize the **text input** as characters: `H e l l o → [H][e][l][l][o]`.
The model learns to map each character to a variable-length stretch of audio frames.

**Problem:** Same character sounds different in different words. `c` in `cat` vs `ceiling` — completely different sounds!

### Approach 2: Phoneme Tokens
Convert text to **phonemes** — the minimal units of sound: `Hello → HH EH L OW`.
Each phoneme is a distinct sound unit with a defined pronunciation.

**Advantage:** Pronunciation is explicit. `c` in `cat` → `K`, `c` in `ceiling` → `S`.
**Problem:** Need a g2p (grapheme-to-phoneme) converter. Different for every language.

### Approach 3: Acoustic Tokens (Codec-based, modern)
Compress the **audio itself** into discrete tokens using a neural codec (e.g., EnCodec, SoundStream).
Audio frames → VQ-VAE codebook → discrete token IDs.

**Advantage:** Language models (like GPT) can directly predict audio tokens — no separate vocoder needed.
**Used in:** VALL-E, AudioLM, SoundStorm.

The key insight: in all approaches, there is an **alignment problem** that NLP doesn't have.

In [ ]:
# ── Demonstrate the three tokenization approaches ──────────────────────────

text = "Hello world"
print('='*60)
print(f'Input text: "{text}"')
print('='*60)

# --- Approach 1: Character tokens ---
char_vocab = {c: i for i, c in enumerate(sorted(set(" abcdefghijklmnopqrstuvwxyz".upper())))
              }
char_tokens = list(text.upper())
char_ids    = [char_vocab.get(c, -1) for c in char_tokens]
print(f'\n[Approach 1] Character tokens:')
print(f'  Tokens: {char_tokens}')
print(f'  IDs:    {char_ids}')
print(f'  Length: {len(char_tokens)} tokens')
print(f'  ⚠ Problem: "C" in cat vs ceiling = same token, different sound!')

# --- Approach 2: Phoneme tokens (using g2p_en) ---
# !pip install g2p_en
try:
    from g2p_en import G2p
    g2p = G2p()
    phonemes = g2p(text)
    phonemes = [p for p in phonemes if p != ' ']
    print(f'\n[Approach 2] Phoneme tokens:')
    print(f'  Phonemes: {phonemes}')
    print(f'  Length:   {len(phonemes)} phonemes')
    print(f'  ✓ Each symbol = one unique sound (ARPAbet standard)')

    # Compare: cat vs ceiling — same 'c', different phonemes
    print(f'\n  Contrast: "cat"     → {g2p("cat")}')
    print(f'            "ceiling" → {g2p("ceiling")}')
    print(f'  → Same letter C, completely different phonemes!')
except ImportError:
    print(f'\n[Approach 2] Phoneme tokens (install g2p_en to run):')
    print(f'  "Hello world" → [HH EH L OW W ER L D]')
    print(f'  "cat"     → [K AE T]  vs  "ceiling" → [S IY L IH NG]')
    print(f'  → Same letter C, different phonemes!')

# --- Approach 3: Acoustic tokens (conceptual demo) ---
print(f'\n[Approach 3] Acoustic tokens (codec-based):')
print(f'  Audio waveform → EnCodec/SoundStream (VQ-VAE) → discrete codes')
print(f'  e.g., 24kHz audio → 75 tokens/sec (at 320 hop) × 8 codebooks')
print(f'  "Hello world" (1s) → ~75 primary tokens + 75×7 refinement tokens')
print(f'  ✓ Can be modeled by a language model (VALL-E, AudioLM)')
print(f'  ✓ No pronunciation dictionary needed — works for any language/sound')
print(f'  ⚠ Very long sequences: 1s audio = 600 tokens total (vs 2 words in NLP)')

## 1.4 The Alignment Problem — Why Speech Has CTC and NLP Doesn't

Even after choosing a tokenization approach, there's still the **alignment problem**:

```
Text:  H  E  L  L  O        ← 5 tokens
Audio: ████████████████████  ← 200 frames

Which frames correspond to which token?
```

In NLP, the input and output are both token sequences of similar length — no alignment needed. In speech, the input (audio frames) and output (text tokens) have very different lengths.

**CTC (Connectionist Temporal Classification)** solves this by:
- Allowing the model to output a special `<blank>` token for "nothing here yet"
- Summing over all valid alignments that collapse to the correct text
- Example: `_H_EE_LL_O_` and `HH_E_L_LO` both collapse to `HELLO`

In [ ]:
# ── Visualize: The Alignment Problem ────────────────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

# NLP: 1-to-1 roughly
ax = axes[0]
nlp_tokens = ['The', 'cat', 'sat']
colors_nlp = ['#4C72B0', '#DD8452', '#55A868']
for i, (tok, col) in enumerate(zip(nlp_tokens, colors_nlp)):
    ax.barh(1, 3, left=i*3, height=0.4, color=col, alpha=0.8)
    ax.text(i*3+1.5, 1, tok, ha='center', va='center', fontsize=12, color='white', fontweight='bold')
    ax.barh(0, 3, left=i*3, height=0.4, color=col, alpha=0.8)
    ax.text(i*3+1.5, 0, tok, ha='center', va='center', fontsize=12, color='white', fontweight='bold')
    ax.annotate('', xy=(i*3+1.5, 0.7), xytext=(i*3+1.5, 0.4),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax.set_xlim(0, 9); ax.set_ylim(-0.3, 1.7)
ax.set_yticks([0, 1]); ax.set_yticklabels(['Output\n(tokens)', 'Input\n(tokens)'])
ax.set_xticks([]); ax.set_title('NLP: Input & Output are both token sequences — alignment is trivial', fontsize=11)

# Speech: M-to-N alignment
ax2 = axes[1]
n_audio_frames = 20
text_tokens = ['H', 'E', 'L', 'L', 'O']
# Simulate alignment: H→frames 0-3, E→4-7, L→8-11, L→12-15, O→16-19
frame_colors = (['#4C72B0']*4 + ['#DD8452']*4 + ['#55A868']*4 +
                ['#C44E52']*4 + ['#9467BD']*4)
token_colors = ['#4C72B0','#DD8452','#55A868','#C44E52','#9467BD']
for i in range(n_audio_frames):
    ax2.barh(1, 1, left=i, height=0.4, color=frame_colors[i], alpha=0.8, edgecolor='white')
for i, (tok, col) in enumerate(zip(text_tokens, token_colors)):
    ax2.barh(0, 1.5, left=i*1.5, height=0.4, color=col, alpha=0.8)
    ax2.text(i*1.5+0.75, 0, tok, ha='center', va='center', fontsize=12, color='white', fontweight='bold')
    mid_frame = i*4 + 2
    ax2.annotate('', xy=(i*1.5+0.75, 0.7), xytext=(mid_frame, 0.4),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1, connectionstyle='arc3,rad=0.3'))
ax2.set_xlim(0, 20); ax2.set_ylim(-0.3, 1.7)
ax2.set_yticks([0, 1]); ax2.set_yticklabels(['Output\n(5 tokens)', 'Input\n(20 frames)'])
ax2.set_xticks([])
ax2.set_title('Speech: Input (audio frames) vs Output (text tokens) have different lengths → CTC needed', fontsize=11)

plt.suptitle('The Alignment Problem: Why Speech Needs CTC and NLP Does Not', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 1.5 How TTS Tokenizes: The Full Pipeline

For TTS (Text-to-Speech), the tokenization flows in the opposite direction:

```
"Hello world"
      ↓  [Text Normalization]
"hello world"  (expand numbers, abbreviations: "Dr." → "Doctor", "5" → "five")
      ↓  [G2P / Character Encoding]
[HH, EH, L, OW, W, ER, L, D]  (phonemes OR characters)
      ↓  [Token Embedding]
Sequence of 512-dim vectors
      ↓  [Tacotron / FastSpeech encoder-decoder]
Mel Spectrogram  (80 bins × T frames)   ← the "image" of speech
      ↓  [Vocoder: WaveNet / HiFi-GAN]
Audio waveform  (16,000 samples/sec)
```

The **duration problem**: each input token must be stretched to cover a variable number of output frames. "Hello" takes ~200ms but "Supercalifragilistic" takes ~800ms.

- **Tacotron**: learns duration implicitly via attention (soft, flexible, but sometimes skips tokens)
- **FastSpeech**: learns duration explicitly via a duration predictor (more stable, fully parallel)

In [ ]:
# ── Build a character-level tokenizer from scratch ──────────────────────────

class SpeechTokenizer:
    """
    Character-level tokenizer for TTS.
    Handles: text normalization, special tokens, emotion tags.
    """
    EMOTIONS = ['[NEUTRAL]', '[HAPPY]', '[SAD]', '[ANGRY]', '[SURPRISED]', '[FEARFUL]']

    def __init__(self):
        # Base vocabulary: printable ASCII
        chars = " !',-.?abcdefghijklmnopqrstuvwxyz"
        self.vocab = {c: i+3 for i, c in enumerate(chars)}
        self.vocab['<PAD>'] = 0
        self.vocab['<BOS>'] = 1
        self.vocab['<EOS>'] = 2
        # Add emotion tag tokens
        for i, e in enumerate(self.EMOTIONS):
            self.vocab[e] = len(self.vocab)
        self.inv_vocab = {v: k for k, v in self.vocab.items()}

    def normalize(self, text):
        """Basic text normalization: lowercase, expand common abbreviations."""
        text = text.lower()
        text = re.sub(r'dr\.', 'doctor', text)
        text = re.sub(r'mr\.', 'mister', text)
        text = re.sub(r'(\d+)', lambda m: self._num_to_words(int(m.group())), text)
        text = re.sub(r'[^a-z !\',\-.?\[\]]', '', text)  # keep only known chars + brackets
        return text.strip()

    def _num_to_words(self, n):
        words = {0:'zero',1:'one',2:'two',3:'three',4:'four',5:'five',
                 6:'six',7:'seven',8:'eight',9:'nine',10:'ten'}
        return words.get(n, str(n))

    def encode(self, text, add_special=True):
        """
        Tokenize text (with optional emotion tags).
        Emotion tags like [HAPPY] are treated as single tokens, not character-by-character.
        """
        # Extract emotion tags first (before normalization)
        tag_pattern = '|'.join(re.escape(e) for e in self.EMOTIONS)
        parts = re.split(f'({tag_pattern})', text)

        tokens = []
        if add_special: tokens.append(self.vocab['<BOS>'])
        for part in parts:
            if part in self.EMOTIONS:
                tokens.append(self.vocab[part])  # emotion tag = single token
            else:
                normalized = self.normalize(part)
                for ch in normalized:
                    if ch in self.vocab:
                        tokens.append(self.vocab[ch])
        if add_special: tokens.append(self.vocab['<EOS>'])
        return tokens

    def decode(self, ids):
        return ''.join(self.inv_vocab.get(i, '?') for i in ids
                       if i not in (self.vocab['<PAD>'], self.vocab['<BOS>'], self.vocab['<EOS>']))

    def __len__(self): return len(self.vocab)


tokenizer = SpeechTokenizer()
print(f'Vocabulary size: {len(tokenizer)}')
print(f'Special tokens: PAD={tokenizer.vocab["<PAD>"]}, BOS={tokenizer.vocab["<BOS>"]}, EOS={tokenizer.vocab["<EOS>"]}')
print(f'Emotion tokens: {[(e, tokenizer.vocab[e]) for e in SpeechTokenizer.EMOTIONS]}')

In [ ]:
# ── Demonstrate tokenization: NLP vs Speech ─────────────────────────────────

test_sentences = [
    "Hello world",
    "Dr. Smith saw 5 patients.",
    "[HAPPY] I got the job!",
    "[SAD] I missed the train.",
    "[ANGRY] This is unacceptable!",
]

print('=' * 65)
print(f'{"Input":35} {"Tokens":6} {"Emotion Tag"}')
print('=' * 65)
for sent in test_sentences:
    ids   = tokenizer.encode(sent)
    back  = tokenizer.decode(ids)
    tag   = [e for e in SpeechTokenizer.EMOTIONS if e in sent]
    print(f'{sent[:33]:35} {len(ids):6}  {tag[0] if tag else "—"}')

print('\n--- Detail: "[HAPPY] I got the job!" ---')
sent = "[HAPPY] I got the job!"
ids = tokenizer.encode(sent)
print(f'Input:   {sent}')
print(f'IDs:     {ids}')
print(f'Decoded: {tokenizer.decode(ids)}')
print(f'\nKey: [HAPPY] = single token ID {tokenizer.vocab["[HAPPY]"]} (not [, H, A, P, P, Y, ])')
print(f'     Just like [CLS] in BERT — one special token = one embedding')

## 1.6 NLP Tokenizer vs Speech Tokenizer — Side-by-Side Comparison

In [ ]:
# ── Visualization: tokenization comparison ──────────────────────────────────

fig, axes = plt.subplots(3, 1, figsize=(14, 8))

sentence = "I got the job!"

# --- NLP BPE tokenization (approximate) ---
nlp_tokens = ['I', 'got', 'the', 'job', '!']
nlp_ids    = [40, 1392, 262, 1693, 0]
ax = axes[0]
colors = plt.cm.Set2(np.linspace(0, 1, len(nlp_tokens)))
x = 0
for tok, tid, col in zip(nlp_tokens, nlp_ids, colors):
    w = len(tok) * 0.8 + 0.4
    ax.barh(0, w, left=x, height=0.5, color=col, edgecolor='white', lw=2)
    ax.text(x+w/2, 0, f'{tok}\n({tid})', ha='center', va='center', fontsize=10, fontweight='bold')
    x += w
ax.set_xlim(0, x+0.5); ax.set_ylim(-0.5, 0.6)
ax.set_yticks([]); ax.set_xticks([])
ax.set_title(f'[NLP] GPT BPE tokenizer: "{sentence}" → 5 tokens, variable width = variable meaning', fontsize=11)

# --- Speech character tokenization ---
chars = list(sentence.lower())
char_ids = [tokenizer.vocab.get(c, -1) for c in chars]
ax2 = axes[1]
colors2 = plt.cm.Pastel1(np.linspace(0, 1, len(chars)))
for i, (ch, cid, col) in enumerate(zip(chars, char_ids, colors2)):
    ax2.barh(0, 1, left=i, height=0.5, color=col, edgecolor='gray', lw=0.5)
    ax2.text(i+0.5, 0, f'{ch}\n({cid})', ha='center', va='center', fontsize=9)
ax2.set_xlim(0, len(chars)+0.5); ax2.set_ylim(-0.5, 0.6)
ax2.set_yticks([]); ax2.set_xticks([])
ax2.set_title(f'[Speech] Character tokenizer: "{sentence}" → {len(chars)} tokens, each character = 1 token', fontsize=11)

# --- Emotion-tagged version ---
sent_emo = f'[HAPPY] {sentence}'
emo_ids  = tokenizer.encode(sent_emo)
emo_disp = [tokenizer.inv_vocab[i] for i in emo_ids]
ax3 = axes[2]
widths = [3.0 if t.startswith('[') else 1.0 for t in emo_disp]
emo_colors_list = ['gold' if t == '[HAPPY]' else
                   ('#2196F3' if t in ('<BOS>','<EOS>') else
                    plt.cm.Pastel1(i/len(emo_disp)))
                   for i, t in enumerate(emo_disp)]
x = 0
for tok, w, col in zip(emo_disp, widths, emo_colors_list):
    ax3.barh(0, w, left=x, height=0.5, color=col, edgecolor='gray', lw=0.5)
    fontsize = 9 if len(tok) <= 3 else 7
    ax3.text(x+w/2, 0, tok, ha='center', va='center', fontsize=fontsize, fontweight='bold' if w>1 else 'normal')
    x += w
ax3.set_xlim(0, x+0.5); ax3.set_ylim(-0.5, 0.6)
ax3.set_yticks([]); ax3.set_xticks([])
ax3.set_title(f'[Emotional Speech] "{sent_emo}" → [HAPPY] = SINGLE token (gold), rest = char tokens', fontsize=11)

plt.suptitle('Three Levels of Speech Tokenization', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

---
# Part 2: Mel Spectrogram — The Bridge Between Tokens and Audio

After tokenization, TTS must produce a Mel Spectrogram. Let's first understand what a spectrogram looks like and then see **how emotion physically changes the spectrogram**.

In [ ]:
# Load a real speech sample and visualize
waveform, sample_rate = torchaudio.load(
    torchaudio.utils.download_asset('tutorial-assets/Lab41-SRI-VOiCES-src-sp0307-ch127535-sg0042.wav')
)
if sample_rate != 16000:
    waveform = T.Resample(sample_rate, 16000)(waveform)
    sample_rate = 16000

mel_tf  = T.MelSpectrogram(sample_rate=16000, n_fft=1024, hop_length=256, n_mels=80)
mel_spec = mel_tf(waveform[0].unsqueeze(0)).squeeze()
log_mel  = torch.log(mel_spec + 1e-9)

fig, axes = plt.subplots(2, 1, figsize=(14, 6))
axes[0].plot(waveform[0].numpy(), linewidth=0.4, color='steelblue')
axes[0].set_title('Raw Waveform (16,000 samples/sec)'); axes[0].set_xlabel('Sample index')

im = axes[1].imshow(log_mel.numpy(), aspect='auto', origin='lower', cmap='magma')
axes[1].set_title('Log Mel Spectrogram (80 bins × time frames)')
axes[1].set_xlabel('Time frames (16ms each)'); axes[1].set_ylabel('Mel frequency bins')
plt.colorbar(im, ax=axes[1], label='Log Energy')
plt.tight_layout(); plt.show()

---
# Part 3: Emotional TTS — Synthesizing Speech with Emotion Tags

## 3.1 Why Emotion in TTS?

Standard TTS (e.g., Google Maps voice) is **neutral** — flat, robotic, no expression. For audiobooks, conversational AI, game NPCs, and accessibility tools, we need TTS that sounds like a human expressing emotion.

## 3.2 How Emotion Tags Work in the TTS Pipeline

*(Based on: Wang et al. 2017 Tacotron + Suno BARK 2023)*

The emotion tag `[HAPPY]` is a **single token** inserted at the start of the input. Its embedding propagates through attention at every layer, conditioning every output mel frame on that emotion style.

```
┌─────────────────────────────────────────────────────────────────┐
│  Input text:   "[HAPPY] I got the job!"                          │
│                                                                  │
│  Tokenize:     [BOS] [HAPPY]  i    g   o   t  ...  !   [EOS]    │
│                  ↓      ↓     ↓    ↓   ↓   ↓         ↓    ↓    │
│  Embed:        1D     512D   512D each character...              │
│                       ↑                                          │
│           Single token, but attention                            │
│           propagates it to ALL positions                         │
│                       ↓                                          │
│  Transformer Encoder (every char attends to [HAPPY])            │
│                       ↓                                          │
│  Mel Decoder → bright, fast mel frames (happy style)            │
│                       ↓                                          │
│  Vocoder (HiFi-GAN) → Audio waveform  🔊                        │
└─────────────────────────────────────────────────────────────────┘
```

> **Key insight:** You don't repeat `[HAPPY]` at every character. One token at position 0 is enough because self-attention lets every subsequent token "see" it at every layer. This is the same mechanism as `[CLS]` in BERT.

## 3.3 How Emotion Changes the Acoustic Signal

Emotion is **not** just about what words you say — it's **how** you say them. Three acoustic dimensions change:

| Emotion | Pitch (F0) | Speaking Rate | Energy | Mel Spectrogram |
|---|---|---|---|---|
| 😐 **Neutral** | Medium, flat | Normal | Medium | Balanced, steady energy |
| 😊 **Happy** | High, rising | **Faster** (+25%) | High | **Bright** — more energy in upper mel bins |
| 😢 **Sad** | Low, falling | **Slower** (−25%) | Low | **Dark** — energy concentrated in lower bins |
| 😠 **Angry** | Very high, variable | Fast, clipped | **Very high** | Harsh, dense, short duration |
| 😨 **Fearful** | High, trembling | Slightly fast | Medium | Uneven energy, high spectral centroid |
| 😲 **Surprised** | **Very high** spike | Normal | Medium-high | Very bright — extreme pitch = high freq energy |

### What this looks like in the Mel Spectrogram:

```
Mel bin
 79 (high freq) ████████████░░░░░   ← HAPPY/SURPRISED: more energy here
                ████████████████████
 40 (mid freq)  ████████████████████
                ████████████████████
  0 (low freq)  ████████████░░░░░░░   ← SAD: more energy here
                |→ time →|→ time →|
                  NEUTRAL    HAPPY (shorter = faster)
```

## 3.4 The Three Levels of Emotion Control in TTS

*(From post-processing → token conditioning → continuous embedding)*

```
Level 1 ── Post-processing  [this lab, Part 3]
  Synthesize neutral → apply speed/pitch shift
  ✓ Simple, no retraining needed
  ✗ Misses: breathiness, vocal fry, laughter timbre

Level 2 ── Conditioning Token  [BARK, this lab Part 3.6]
  [HAPPY] token → embedding → conditions decoder at every step
  ✓ Natural-sounding, model-learned style
  ✗ Needs labelled emotion training data per language

Level 3 ── Continuous Style Embedding  [StyleTTS2, ElevenLabs]
  Emotion = continuous vector → interpolate between emotions
  ✓ Full control, can blend (70% sad + 30% angry)
  ✗ Complex training, needs reference audio per style
```

**Reference:**
- BARK prompt engineering (Suno, 2023): https://github.com/suno-ai/bark#-usage
- StyleTTS2 (Li et al., 2023): https://arxiv.org/abs/2306.07691

---

In [ ]:
import urllib.request, zipfile, os, glob
import torchaudio
import torchaudio.transforms as T
from IPython.display import Audio, display
import numpy as np
import matplotlib.pyplot as plt

os.makedirs('data/ravdess', exist_ok=True)

# ── Download RAVDESS ──────────────────────────────────────────────────────
# RAVDESS: Ryerson Audio-Visual Database of Emotional Speech and Song
# Livingstone & Russo (2018). PLOS ONE. https://doi.org/10.1371/journal.pone.0196391
# 24 actors (12 male, 12 female), 8 emotions, 2 intensities, ~1440 clips
#
# Full dataset = 24.8 GB (video). We use the speech-only audio subset (~215 MB).
# Available via Zenodo (open access): https://zenodo.org/record/1188976

RAVDESS_URL = 'https://zenodo.org/record/1188976/files/Audio_Speech_Actors_01-24.zip'

zip_path = 'data/ravdess/ravdess_speech.zip'
if not os.path.exists('data/ravdess/Actor_01'):
    print('Downloading RAVDESS Speech Audio (~215 MB)...')
    print('(This may take a few minutes on a slow connection)')
    urllib.request.urlretrieve(RAVDESS_URL, zip_path)
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('data/ravdess/')
    print('Done!')
else:
    print('RAVDESS already downloaded.')

# ── RAVDESS filename convention ────────────────────────────────────────────
# Format: 03-01-XX-XX-XX-XX-XX.wav
# Modality:    03 = audio-only
# Channel:     01 = speech
# Emotion:     01=neutral 02=calm 03=happy 04=sad 05=angry 06=fearful 07=disgust 08=surprised
# Intensity:   01=normal  02=strong
# Statement:   01="Kids are talking by the door"  02="Dogs are sitting by the door"
# Repetition:  01 or 02
# Actor:       01-24 (odd=male, even=female)

EMOTION_MAP = {
    '01': 'neutral', '02': 'calm', '03': 'happy', '04': 'sad',
    '05': 'angry',   '06': 'fearful', '07': 'disgust', '08': 'surprised'
}

def parse_ravdess_filename(path):
    parts = os.path.basename(path).replace('.wav','').split('-')
    return {
        'emotion':   EMOTION_MAP.get(parts[2], 'unknown'),
        'intensity': 'normal' if parts[3] == '01' else 'strong',
        'statement': 'Kids...' if parts[4] == '01' else 'Dogs...',
        'actor':     int(parts[6]),
        'gender':    'male' if int(parts[6]) % 2 == 1 else 'female'
    }

# ── Load all audio files with metadata ────────────────────────────────────
all_files = sorted(glob.glob('data/ravdess/Actor_*/*.wav'))
data = []
for path in all_files:
    meta = parse_ravdess_filename(path)
    meta['path'] = path
    data.append(meta)

import pandas as pd
df = pd.DataFrame(data)
print(f'Total clips: {len(df)}')
print(f'Emotions: {df.emotion.unique()}')
print(f'Actors: {df.actor.nunique()} ({df[df.gender=="male"].actor.nunique()} male, {df[df.gender=="female"].actor.nunique()} female)')
print()
print(df.groupby('emotion').size().to_string())


In [ ]:
# ── Load one clip per emotion and compute mel spectrograms ───────────────
MEL_SR = 22050
mel_tf = T.MelSpectrogram(sample_rate=MEL_SR, n_fft=1024, hop_length=256, n_mels=80)

EMOTIONS_SHOW = ['neutral', 'happy', 'sad', 'angry', 'fearful', 'surprised']
emotion_audio = {}   # emotion → (waveform, sr)
emotion_mels  = {}   # emotion → log_mel tensor

print('Loading one clip per emotion (normal intensity, Actor 01)...')
for emo in EMOTIONS_SHOW:
    # Pick actor 01, normal intensity for consistency
    subset = df[(df.emotion == emo) & (df.intensity == 'normal') & (df.actor == 1)]
    if len(subset) == 0:
        subset = df[(df.emotion == emo) & (df.intensity == 'normal')]
    if len(subset) == 0:
        subset = df[df.emotion == emo]
    row = subset.iloc[0]
    wvf, sr = torchaudio.load(row['path'])
    if sr != MEL_SR:
        wvf = T.Resample(sr, MEL_SR)(wvf)
    mel = mel_tf(wvf[0].unsqueeze(0)).squeeze()
    emotion_audio[emo] = (wvf, MEL_SR)
    emotion_mels[emo]  = torch.log(mel + 1e-9)
    dur = wvf.shape[1] / MEL_SR
    print(f'  [{emo:12}] {os.path.basename(row.path):50} {dur:.2f}s')

# ── Listen ────────────────────────────────────────────────────────────────
print()
print('=== Real RAVDESS Emotional Speech ===')
print('Text: "Kids are talking by the door" (same sentence, different emotions)')
print()
for emo in EMOTIONS_SHOW:
    wvf, sr = emotion_audio[emo]
    print(f'[{emo.upper()}]')
    display(Audio(wvf.numpy(), rate=sr))


In [ ]:
# ── Mel Spectrogram Grid: Real Emotional Speech ──────────────────────────
EMOTION_COLORS = {'neutral':'gray','happy':'gold','sad':'royalblue',
                  'angry':'crimson','fearful':'purple','surprised':'orange'}

fig, axes = plt.subplots(2, 3, figsize=(18, 8))
for ax, emo in zip(axes.flatten(), EMOTIONS_SHOW):
    log_mel = emotion_mels[emo]
    wvf, sr = emotion_audio[emo]
    dur = wvf.shape[1] / sr

    im = ax.imshow(log_mel.numpy(), aspect='auto', origin='lower',
                   cmap='magma', vmin=-10, vmax=2)
    ax.set_title(f'[{emo.upper()}]  ({dur:.2f}s)',
                 fontsize=12, color=EMOTION_COLORS[emo], fontweight='bold')
    ax.set_xlabel('Time frames (11.6ms each)')
    ax.set_ylabel('Mel bins')
    ax.text(0.98, 0.02, f'{dur:.2f}s', transform=ax.transAxes,
            ha='right', va='bottom', color='white', fontsize=10)

plt.suptitle(
    'Real RAVDESS Speech — Same Sentence, 6 Emotions\n'
    '"Kids are talking by the door" (Actor 01, normal intensity)',
    fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

print('What to observe in REAL emotional speech (vs synthetic pitch-shift):')
print('  NEUTRAL   → steady energy, even formant structure')
print('  HAPPY     → slightly brighter, faster tempo visible as fewer frames')
print('  SAD       → more frames (slower), energy concentrated low')
print('  ANGRY     → very bright upper bands, dense energy throughout')
print('  FEARFUL   → irregular energy bursts (breathiness / glottal stops)')
print('  SURPRISED → short, high-energy burst at start (the gasp/intake)')


## 3.4 Acoustic Feature Analysis: Real vs Synthetic Emotion

Now let's **quantify** what we hear — and compare real RAVDESS emotion features
to what a naive pitch-shift/speed-shift would produce.
This shows why real emotional speech data matters.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cols = [EMOTION_COLORS[e] for e in EMOTIONS_SHOW]

durations, energies, centroids = [], [], []
for emo in EMOTIONS_SHOW:
    wvf, sr = emotion_audio[emo]
    durations.append(wvf.shape[1] / sr)
    energies.append(wvf.pow(2).mean().sqrt().item() * 1000)
    mel = emotion_mels[emo]
    centroid = (torch.arange(80).float() * mel.exp().mean(1)).sum() / mel.exp().mean(1).sum()
    centroids.append(centroid.item())

for ax, vals, title, ylabel in zip(axes,
    [durations, energies, centroids],
    ['Duration (s)\n[Real speech: sad longest, angry shortest]',
     'RMS Energy × 1000\n[Angry loudest, sad quietest]',
     'Spectral Centroid (mel bin)\n[Happy/Surprised bright, Sad dark]'],
    ['seconds', 'RMS × 1000', 'mel bin (0=low, 79=high)']):
    bars = ax.bar(EMOTIONS_SHOW, vals, color=cols, edgecolor='black', linewidth=0.7)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01*max(vals),
                f'{v:.2f}', ha='center', fontsize=8)
    ax.set_title(title, fontsize=10)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis='x', rotation=25)

plt.suptitle('Acoustic Fingerprint — Real RAVDESS Emotional Speech',
             fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

# ── Side-by-side waveforms ────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 6))
for ax, emo in zip(axes.flatten(), EMOTIONS_SHOW):
    wvf, sr = emotion_audio[emo]
    t = np.linspace(0, wvf.shape[1]/sr, wvf.shape[1])
    ax.plot(t, wvf[0].numpy(), linewidth=0.4, color=EMOTION_COLORS[emo])
    ax.set_title(f'[{emo.upper()}]', color=EMOTION_COLORS[emo], fontweight='bold')
    ax.set_xlabel('Time (s)'); ax.set_ylabel('Amplitude')
    ax.set_ylim(-1, 1)
plt.suptitle('Raw Waveforms — Real RAVDESS Emotional Speech', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()


## 3.5 BARK — Native Emotion via Acoustic Tokens

BARK (Suno, 2023) goes beyond pitch-shifting. It generates genuine paralinguistic features:
actual **laughter**, **sighing breath**, **shouting timbre** — because it models audio as
discrete acoustic tokens (like GPT predicting text), trained on real expressive speech.

| RAVDESS (ground truth) | Post-processing (Level 1) | BARK (Level 2) |
|---|---|---|
| Actor *performs* emotion | Neutral → pitch-shifted | LM predicts emotion tokens |
| Real breathiness, glottal stops | Robotic — just pitch/speed | Natural paralinguistics |
| Fixed vocabulary (24 actors) | Any text, any speed | Any text, any prompt tag |

BARK prompt tags: `[laughs]` `[sighs]` `[clears throat]` `[gasps]` `[whispers]` `[MAN]` `[WOMAN]`


In [ ]:
# !pip install bark
try:
    from bark import SAMPLE_RATE, generate_audio, preload_models
    preload_models()
    BARK_AVAILABLE = True
    print('BARK loaded!')
except ImportError:
    BARK_AVAILABLE = False
    print('BARK not installed. pip install bark to enable.')

bark_prompts = {
    'neutral':   "Kids are talking by the door.",
    'happy':     "[laughs] Kids are talking by the door!",
    'sad':       "[sighs] Kids are talking by the door...",
    'angry':     "[shouts] KIDS are talking by the door!",
    'surprised': "[gasps] Kids are talking by the door?!",
    'whisper':   "[whispers] Kids are talking by the door.",
}

if BARK_AVAILABLE:
    import scipy.io.wavfile as wav
    os.makedirs('data/speech/bark', exist_ok=True)
    for emotion, prompt in bark_prompts.items():
        print(f'Generating [{emotion}]...')
        audio = generate_audio(prompt)
        wav.write(f'data/speech/bark/{emotion}.wav', SAMPLE_RATE, audio)
        display(Audio(audio, rate=SAMPLE_RATE))
else:
    print('Showing BARK prompt examples (install bark to generate):')
    for emo, prompt in bark_prompts.items():
        print(f'  [{emo:12}]: {prompt}')
    print()
    print('Note: Compare BARK output with RAVDESS clips above.')
    print('BARK [laughs] generates real laughter acoustics, not just a faster/higher version.')


---
## Exercises

### Exercise 1: Speech vs NLP Tokenization — Building Intuition

a) Using the `SpeechTokenizer` defined in Part 1, tokenize the following sentences and fill in the table:

```python
sentences = [
    "Hello, how are you?",
    "Dr. Smith prescribed 10 tablets.",
    "[HAPPY] I got the job!",
    "[SAD] I lost my wallet.",
    "[ANGRY] This is completely unacceptable!",
]
```

| Sentence | # Char tokens | # Tokens (with BOS/EOS) | Emotion tag token ID |
|---|---|---|---|
| Hello, how are you? | ? | ? | — |
| Dr. Smith... | ? | ? | — |
| [HAPPY] I got the job! | ? | ? | ? |

b) In the tokenizer's `normalize()` method, `"Dr. Smith"` becomes `"doctor smith"`. Why is this normalization critical for TTS? What would happen if the model received `"Dr."` as input?

c) In NLP, `[CLS]` is a special token that acts as a "summary" of the sentence. In speech, the emotion tag `[HAPPY]` is also a single special token. Explain the **architectural similarity** — how does a single token at the beginning of a sequence influence the model's entire output?

---

### Exercise 2: Spectrogram Emotion Fingerprinting

a) Compute the following features for each of the 6 emotions synthesized in Part 3:

```python
# For each emotion:
# 1. Duration (seconds)
# 2. RMS Energy
# 3. Mel centroid (frequency center of mass)
# 4. Temporal energy variance (how much does energy fluctuate over time?)
```

Plot all 4 features as a radar chart (one polygon per emotion). Which emotion has the most distinctive acoustic fingerprint?

b) Take the mel spectrograms of `[HAPPY]` and `[SAD]`. Compute the pixel-wise difference (SAD - HAPPY). What parts of the frequency spectrum differ the most? Plot the difference map.

c) Would you expect a neural network to be able to classify emotion from just the mel spectrogram? Train a simple CNN on the 6 emotion spectrograms (you can use the augmented versions). What accuracy do you get with a leave-one-out cross-validation?

---

### Exercise 3: Extend the Tokenizer — Prosody Tags

Real TTS systems like Amazon Polly use **SSML (Speech Synthesis Markup Language)** which supports rich prosody control:

```xml
<speak>
  <prosody rate="slow" pitch="low">I'm very tired.</prosody>
  <break time="500ms"/>
  <emphasis level="strong">This is important!</emphasis>
</speak>
```

a) Extend `SpeechTokenizer` to support the following additional special tokens:
- `[SLOW]`, `[FAST]`, `[NORMAL]` — speaking rate
- `[HIGH]`, `[LOW]` — pitch
- `[PAUSE]` — short pause
- `[EMPHASIS]` — word emphasis

b) Tokenize these sentences and check the output:
```
"[SLOW] This is very [EMPHASIS] important [PAUSE] please listen carefully."
"[HAPPY] [FAST] Let's go! I'm so excited!"
```

c) How do combined tags (both emotion + prosody) interact? Could `[HAPPY][SLOW]` be a valid combination? When might you use it?

---

### Exercise 4 (Challenge): Mixed-Emotion Synthesis

Real human speech is rarely one single emotion. A sentence like *"I'm fine"* can be happy, sarcastic, or fearful depending on context.

a) Using the Coqui-TTS API, synthesize the same sentence 3 times and apply **different post-processing** to blend emotions:

```python
# Blend: 70% sad + 30% angry (by interpolating the acoustic features)
def blend_emotions(wvf1, wvf2, alpha=0.7):
    # Ensure same length by trimming/padding
    min_len = min(wvf1.shape[1], wvf2.shape[1])
    return alpha * wvf1[:, :min_len] + (1 - alpha) * wvf2[:, :min_len]
```

b) Compare mel spectrograms of:
- Pure `[SAD]`
- Pure `[ANGRY]`
- 70% SAD + 30% ANGRY blend

Does the blend spectrogram look like a visual average of the two?

c) Listen to the blend. Does it sound natural? Why or why not? (Hint: naively blending audio waveforms is not the same as blending acoustic style — what would you need to do differently?)

d) Research question: **ElevenLabs** allows users to control emotion via a "stability" and "similarity boost" slider. Based on what you learned in this lab, what acoustic dimensions do you think these sliders are controlling?

---

### The Report
Submit on **Teal Classroom** before the deadline. Include:
- The tokenization comparison table from Exercise 1
- Radar chart and spectrogram difference map from Exercise 2
- Extended tokenizer code from Exercise 3
- Audio samples + blend spectrograms from Exercise 4
- Discussion: How does understanding speech tokenization change how you think about training a TTS model?

---